In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from utils import targetEncode, labelEncode, oneHotEncode

from sklearn.ensemble import IsolationForest
from sklearn import GaussianMixture


In [ ]:
# Pipeline Variables
bedford_threshold = .235
z_threshold = 3.0
gmmComponents = 10
tSteps = .2

# Clustering Functions

In [ ]:
# GMM Clustering
class GMM():
    def __init__(self,init_data: pd.DataFrame, n_components = 10):
        self.algo = GaussianMixture(n_components=n_components, random_state=42, init_params='kmeans')
        self.algo.fit(init_data)
        self.known_data = init_data.copy()
    
    def get_anomalous(self, data: pd.DataFrame, threshold: float = .1):
        predicted_probabilities = self.algo.predict_proba(data)
        return predicted_probabilities

    def update_known(self, data: pd.DataFrame):
        self.known_data = pd.concat(self.known_data, data)
        self.algo.fit(self.known_data)
    



In [ ]:
# IF Clustering
class IF(): 
    def __init__(self,init_data: pd.DataFrame, n_components = 10):
        self.algo = IsolationForest(n_estimators=200, )
        self.known_data = init_data.copy()
    
    def get_anomalous(self, data: pd.DataFrame, threshold: float = .1):
        predicted_probabilities = self.algo.predict(data)
        return predicted_probabilities

    def update_known(self, data: pd.DataFrame):
        self.known_data = pd.concat(self.known_data, data.copy())
        self.algo.fit(self.known_data)



This workbook outlines the programatic flow of the Anomily Detection Pipeline.

Sections include:
- Data Cleaning and Preperation
    - Clean, Engineer and Encode Data for processing
    - Simulated streaming via batching
- Anomily Detection Pipeline
    - Statistical Analysis Smoke Test
        - Benfords Law
        - Parameter Slope (Sudden changes in total or number of transactions triggers review)
    - Anomaly Segmentation Loop
        - GMM indentifies records that are more like observed anomalies than `normal` data.
        - IF identifies the items most isolated from eachoter.
        - Intersection is population to be removed.
    - Statistical Retest
        - If statistically normal
            - approve remaining transactions as normal
                - store in memory with decaying term (this term can be hypertuned)
            - Add anomalious transactions to anomaly cluster.
                - does not have a decaying term. but may be updated to maximize GMM representation.
        - If still statistically suspissious
            - increment thresholds
                - if threshold exceeds 75% flag entire batch as suspissious and system alert for T1 support to confirm fraud event
            - repete Anomaly Segmentation Loop


In [2]:
# Load Data
engData = pd.read_csv('./data/eng_data.csv')
print(engData.head().to_markdown())

|    | Timestamp           |   TransactionID |   AccountID |   Amount | Merchant   | TransactionType   | Location    |   Year |   Quarter |   Month | Day_Of_Week   |   Day_Of_Month |   Day_Of_Year |   Hour |   Minute |
|---:|:--------------------|----------------:|------------:|---------:|:-----------|:------------------|:------------|-------:|----------:|--------:|:--------------|---------------:|--------------:|-------:|---------:|
|  0 | 2023-01-01 08:00:00 |            1127 |           4 | 95071.9  | H          | Purchase          | Tokyo       |   2023 |         1 |       1 | Sunday        |              1 |             1 |      8 |        0 |
|  1 | 2023-01-01 08:01:00 |            1639 |          10 | 15607.9  | H          | Purchase          | London      |   2023 |         1 |       1 | Sunday        |              1 |             1 |      8 |        1 |
|  2 | 2023-01-01 08:02:00 |             872 |           8 | 65092.3  | E          | Withdrawal        | London      |   202

In [17]:
# Target Encoding
preppedData = targetEncode(data=engData, encodeFeatures=['TransactionType', 'Location', 'Day_Of_Week', 'Merchant'], targetFeature='Amount')
preppedData['Timestamp'] = pd.to_datetime(preppedData['Timestamp'])
preppedData = preppedData.set_index('Timestamp')
print(preppedData.head().to_markdown())


| Timestamp           |   TransactionID |   AccountID |   Amount |   Merchant |   TransactionType |   Location |   Year |   Quarter |   Month |   Day_Of_Week |   Day_Of_Month |   Day_Of_Year |   Hour |   Minute |
|:--------------------|----------------:|------------:|---------:|-----------:|------------------:|-----------:|-------:|----------:|--------:|--------------:|---------------:|--------------:|-------:|---------:|
| 2023-01-01 08:00:00 |            1127 |           4 | 95071.9  |    50455.3 |           50243.5 |    50393.6 |   2023 |         1 |       1 |       50165.7 |              1 |             1 |      8 |        0 |
| 2023-01-01 08:01:00 |            1639 |          10 | 15607.9  |    50455.3 |           50243.5 |    50241.7 |   2023 |         1 |       1 |       50165.7 |              1 |             1 |      8 |        1 |
| 2023-01-01 08:02:00 |             872 |           8 | 65092.3  |    50217.7 |           50141.6 |    50241.7 |   2023 |         1 |       1 |     

In [ ]:
from scipy.stats import wasserstein_distance

def is_Anomalous(data: pd.DataFrame):
    batch = data.copy()
    batch['first_digit'] = batch['Amount'].astype(str).str[0].astype(int)
    observed_counts = batch['first_digit'].value_counts().sort_index()
    benford_expected_proportions = pd.Series([np.log10(1 + 1/d) for d in range(1, 10)], index=range(1, 10))
    observed_proportions = observed_counts / len(batch)
    distance = np.linalg.norm(observed_proportions - benford_expected_proportions)
    return distance > bedford_threshold

def big_Slope(data: pd.DataFrame):
    # Batch by account and merchant, define threshold for distorical divergence
    return False

seenData = pd.DataFrame(columns=[*preppedData.columns, 'Decay'])
anomalies = pd.DataFrame(columns=preppedData.columns)
bedford_threshold = .235
i = 0
anomaly_threshold = .1
sample_period = '5D'


# Prime Pump:

for timestamp, data in preppedData.resample(sample_period):
        gmmModel = GMM(data)
        ifModel = IF(data)
        break;

# Time Batching
# print(len(preppedData))
# print(benford_expected_proportions)
for timestamp, data in preppedData.resample(sample_period):
    detectedAnomalies = pd.DataFrame()
    analyzedData = data.copy()
    while(len(analyzedData) and (is_Anomalous(analyzedData) or big_Slope(analyzedData))):
        gmmModel = GMM()
        ifModel = IF()

        # Get Z-Score outliers
        print('Anomalies Detected')
        gmmResults = gmmModel

        anomaly_threshold += .1
    
    seenData['Decay'] = seenData['Decay'] - .2
    seenData.drop(seenData[seenData['Decay'] == 0].index, inplace=True)
    newSeen = analyzedData.copy()
    newSeen['Decay'] = 1
    seenData = pd.concat(seenData, newSeen)


print(i)


Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
Anomalies Detected
14


In [ ]:

print(min(distances))
print(max(distances))

0.21279226697896517
0.2571964047299128
